# 通过API获取加州的数据，并保存到文件中

## 步骤一.导入所有库

In [1]:
import requests
import pandas as pd
import os

print("导入库完成！")

导入库完成！


## 步骤二.定义常量和参数

In [2]:
# SODA2 API 端点
BASE_URL = "https://data.transportation.gov/resource/85tf-25kj.json"

# SoQL 筛选器，用于仅获取加州数据
STATE_FILTER = "statename='CALIFORNIA'"

# API 分页设置：一次获取1000条记录
PAGE_SIZE = 1000

# 定义输出路径
# 我们在 'notebooks' 目录中，所以需要用 '..' 返回上一级，然后进入 'data/raw'
OUTPUT_DIR = os.path.join('..', 'data', 'raw')
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'california_raw_data.csv')

## 步骤三.循环调用API获取所有数据

In [3]:
# 用于存储所有获取到的记录
all_records = []
offset = 0
page = 1

print("开始从FRA API获取加州数据...")

while True:
    # 构造带有筛选和分页参数的请求
    params = {
        '$where': STATE_FILTER,
        '$limit': PAGE_SIZE,
        '$offset': offset
    }
    
    try:
        response = requests.get(BASE_URL, params=params)
        
        # 检查请求是否成功
        if response.status_code != 200:
            print(f"错误：API 请求失败，状态码: {response.status_code}")
            print(f"响应内容: {response.text}")
            break
            
        batch_data = response.json()
        
        # 如果返回的数据为空，说明已经获取了所有页面
        if not batch_data:
            print("已成功获取所有数据页。")
            break
            
        # 将这批数据添加到总列表中
        all_records.extend(batch_data)
        print(f"已获取第 {page} 页数据 ({len(batch_data)} 条记录)...")
        
        # 准备获取下一页
        offset += PAGE_SIZE
        page += 1
        
    except requests.exceptions.RequestException as e:
        print(f"网络请求错误: {e}")
        break
    except Exception as e:
        print(f"处理数据时发生错误: {e}")
        break

print(f"数据获取完成。总共获取 {len(all_records)} 条加州事故记录。")

开始从FRA API获取加州数据...
已获取第 1 页数据 (1000 条记录)...
网络请求错误: HTTPSConnectionPool(host='data.transportation.gov', port=443): Max retries exceeded with url: /resource/85tf-25kj.json?%24where=statename%3D%27CALIFORNIA%27&%24limit=1000&%24offset=1000 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1000)')))
数据获取完成。总共获取 1000 条加州事故记录。


## 步骤 4：将数据转换为 DataFrame 并保存

In [5]:
if all_records:
    # 1. 将数据转换为 DataFrame
    df = pd.DataFrame(all_records)
    
    # 2. 确保 'data/raw' 目录存在
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # 3. 将 DataFrame 保存到 CSV 文件
    try:
        df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
        print(f"数据已成功保存到: {OUTPUT_FILE}")
    except IOError as e:
        print(f"保存文件失败: {e}")
else:
    print("未获取到任何数据，不创建文件。")

数据已成功保存到: ..\data\raw\california_raw_data.csv


## 步骤五：验证数据

In [6]:
if 'df' in locals() and not df.empty:
    print("\n数据验证：")
    print(f"DataFrame 形状 (行, 列): {df.shape}")
    print("\n数据前5行预览:")
    print(df.head())
else:
    print("无数据可供验证。")


数据验证：
DataFrame 形状 (行, 列): (11383, 161)

数据前5行预览:
  reportingrailroadcode                reportingrailroadname  year  \
0                  BNSF                 BNSF Railway Company  1997   
1                  BNSF                 BNSF Railway Company  1999   
2                  BNSF                 BNSF Railway Company  2015   
3                  SJVR  San Joaquin Valley Railroad Company  2011   
4                  BNSF                 BNSF Railway Company  2011   

  accidentnumber                                                url  \
0      SC1097111  {'url': 'https://safetydata.fra.dot.gov/Office...   
1      SC0699123  {'url': 'https://safetydata.fra.dot.gov/Office...   
2      CA0815200  {'url': 'https://safetydata.fra.dot.gov/Office...   
3       ID110340  {'url': 'https://safetydata.fra.dot.gov/Office...   
4      CA0911203  {'url': 'https://safetydata.fra.dot.gov/Office...   

  accidentyear accidentmonth maintenancerailroadcode maintenancerailroadname  \
0           97       